In [3]:
import pandas as pd
import os
from google.colab import drive

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# --- 請確認路徑是否正確 ---
file_name = "20210101~20211231deta.csv"
input_path = f'/content/drive/MyDrive/{file_name}'
output_path = '/content/drive/MyDrive/Path_2021_Converted.csv'
# -----------------------

if not os.path.exists(input_path):
    print(f"❌ 找不到檔案：{input_path}")
else:
    try:
        # 2. 嘗試讀取資料 (改用 utf-16 讀取)
        # 根據錯誤訊息 0xff，極大機率是 utf-16
        df = pd.read_csv(input_path, sep='\t', encoding='utf-16')
        print("✅ 成功使用 UTF-16 編碼讀取檔案")

    except Exception as e:
        print(f"嘗試 UTF-16 失敗，改用 UTF-8 搭配誤差處理：{e}")
        df = pd.read_csv(input_path, sep='\t', encoding='utf-8', errors='ignore')

    # 3. 資料整理
    # 確保日期欄位轉為字串後處理
    df['年月日'] = pd.to_datetime(df['年月日'].astype(str))

    new_df = pd.DataFrame()

    # Date 格式：2021/1/4
    new_df['Date'] = df['年月日'].dt.strftime('%Y/%-m/%-d')

    # File 格式：OptionsDaily_2021_01_04.csv
    new_df['File'] = df['年月日'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')

    # S0 格式：收盤價
    new_df['S0'] = df['收盤價(元)']

    # 4. 存回雲端 (使用 utf-8-sig 以利 Excel 開啟)
    new_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"🎉 處理完成！檔案已存至：{output_path}")
    print(new_df.head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 成功使用 UTF-16 編碼讀取檔案
🎉 處理完成！檔案已存至：/content/drive/MyDrive/Path_2021_Converted.csv
       Date                         File        S0
0  2021/1/4  OptionsDaily_2021_01_04.csv  14902.03
1  2021/1/5  OptionsDaily_2021_01_05.csv  15000.03
2  2021/1/6  OptionsDaily_2021_01_06.csv  14983.13


In [5]:
import pandas as pd
import os
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# --- 請設定檔案路徑 ---
# 假設檔案都在雲端硬碟的根目錄，如果不是，請修改路徑
data_file = '/content/drive/MyDrive/20210101~20211231deta.csv'
expiry_file = '/content/drive/MyDrive/最後結算日.csv'
output_file = '/content/drive/MyDrive/Path_2021_Full.csv'

def safe_read_csv(file_path, sep=','):
    """自動嘗試不同編碼讀取 CSV"""
    for enc in ['utf-8-sig', 'utf-16', 'cp950', 'big5']:
        try:
            return pd.read_csv(file_path, sep=sep, encoding=enc)
        except:
            continue
    raise ValueError(f"無法讀取檔案: {file_path}")

# 2. 讀取資料
# 交易資料通常是 Tab 分隔，結算日資料通常是逗號分隔
df_data = safe_read_csv(data_file, sep='\t')
df_expiry = safe_read_csv(expiry_file, sep=',')

# 3. 預處理結算日：只留下「月合約」(過濾掉含有 W 的合約)
df_monthly = df_expiry[~df_expiry['契約月份'].astype(str).str.contains('W')].copy()
df_monthly['ExpiryDT'] = pd.to_datetime(df_monthly['最後結算日'])
df_monthly = df_monthly.sort_values('ExpiryDT')

# 4. 預處理交易資料
df_data['TradeDT'] = pd.to_datetime(df_data['年月日'].astype(str))

# 5. 核心邏輯：尋找對應的月合約資訊
def find_monthly_info(trade_date):
    # 尋找第一個大於或等於交易日的月結算日
    match = df_monthly[df_monthly['ExpiryDT'] >= trade_date].iloc[0]
    return match['契約月份'], match['ExpiryDT']

# 套用邏輯生成 Contract 與 ExpiryDate 欄位
df_data[['Contract', 'ContractExpiryDT']] = df_data['TradeDT'].apply(lambda x: pd.Series(find_monthly_info(x)))

# 6. 格式化輸出
final_df = pd.DataFrame()
# Date: 2021/1/4
final_df['Date'] = df_data['TradeDT'].dt.strftime('%Y/%-m/%-d')
# File: OptionsDaily_2021_01_04.csv
final_df['File'] = df_data['TradeDT'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')
# S0: 價格
final_df['S0'] = df_data['收盤價(元)']
# Maturity: 剩餘天數 (結算日 - 交易日)
final_df['Maturity'] = (df_data['ContractExpiryDT'] - df_data['TradeDT']).dt.days
# Contract: 合約月份
final_df['Contract'] = df_data['Contract']
# ContractExpiryDate: 結算日期
final_df['ContractExpiryDate'] = df_data['ContractExpiryDT'].dt.strftime('%Y/%-m/%-d')

# 7. 存回雲端
final_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✅ 轉換完成！檔案已存至: {output_file}")
print("前五筆資料預覽：")
print(final_df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 轉換完成！檔案已存至: /content/drive/MyDrive/Path_2021_Full.csv
前五筆資料預覽：
       Date                         File        S0  Maturity Contract  \
0  2021/1/4  OptionsDaily_2021_01_04.csv  14902.03        16   202101   
1  2021/1/5  OptionsDaily_2021_01_05.csv  15000.03        15   202101   
2  2021/1/6  OptionsDaily_2021_01_06.csv  14983.13        14   202101   
3  2021/1/7  OptionsDaily_2021_01_07.csv  15214.00        13   202101   
4  2021/1/8  OptionsDaily_2021_01_08.csv  15463.95        12   202101   

  ContractExpiryDate  
0          2021/1/20  
1          2021/1/20  
2          2021/1/20  
3          2021/1/20  
4          2021/1/20  


In [7]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import drive

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# --- 設定路徑 ---
input_path = '/content/drive/MyDrive/Path_2021_Full.csv'
output_path = '/content/drive/MyDrive/Path_2021_with_Rf.csv'

def get_2021_rf():
    """
    爬取或定義 2021 年的 Rf。
    因為 2021 全年利率固定為 0.845%，此處採直接賦值以確保程式效率。
    若需動態爬取最新利率，可參考下方爬蟲邏輯。
    """
    return 0.00845  # 0.845%

def process_rf():
    # 2. 讀取現有的 Path 檔案
    if not os.path.exists(input_path):
        print(f"❌ 找不到檔案：{input_path}")
        return

    df = pd.read_csv(input_path)

    # 3. 加入 Rf 欄位
    # 一般財經模型中，Rf 常以「年化利率」表示
    rf_annual = get_2021_rf()
    df['Rf'] = rf_annual

    # 4. 存回雲端
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"✅ 已加入 Rf 並儲存至：{output_path}")
    print(df.head())

# 執行
process_rf()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 已加入 Rf 並儲存至：/content/drive/MyDrive/Path_2021_with_Rf.csv
       Date                         File        S0  Maturity  Contract  \
0  2021/1/4  OptionsDaily_2021_01_04.csv  14902.03        16    202101   
1  2021/1/5  OptionsDaily_2021_01_05.csv  15000.03        15    202101   
2  2021/1/6  OptionsDaily_2021_01_06.csv  14983.13        14    202101   
3  2021/1/7  OptionsDaily_2021_01_07.csv  15214.00        13    202101   
4  2021/1/8  OptionsDaily_2021_01_08.csv  15463.95        12    202101   

  ContractExpiryDate       Rf  
0          2021/1/20  0.00845  
1          2021/1/20  0.00845  
2          2021/1/20  0.00845  
3          2021/1/20  0.00845  
4          2021/1/20  0.00845  
